In [185]:
import numpy as np
np.random.seed(203)
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
sns.set(style="whitegrid")
from collections import Counter

from datetime import datetime
# from sklearn.manifold import TSNE

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, MinMaxScaler, KBinsDiscretizer
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, precision_recall_curve
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
import joblib
# from sklearn.decomposition import PCA
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# from sklearn.linear_model import  LogisticRegression
# import xgboost as xgb
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# from imblearn.over_sampling import RandomOverSampler
# from imblearn.under_sampling import RandomUnderSampler

from keras.models import Sequential

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from collections import Counter
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, SpatialDropout1D
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

from keras.layers import Input, Dense
from keras.models import Model, Sequential
from keras import regularizers

In [186]:
def quantile_label(quantile_split, feature_values, duplicates=False):
#Function to define the lower & upper limits after pd.qcut (quantile cut)
#applicable for numeric (continue) feature
#INPUT :
#quantile_split: list of intervals of split; for example quantile_split = [0, 0.25, 0.5, 0.75, 1]
#feature_values : all values of the feature to split (for example : feature_values = df['age'])

#duplicates : if there is one duplicate in qantile cut, the bins numbre reduce to len(quantile_split) - 2
#else : no duplicate : n_bins = len(quantile_split)-1


    feature_quantile = feature_values.quantile(quantile_split)
        #lower_limit = quantile_split[i]
        #upper_limit = quantile_split[i+1]
        #print(str(int(feature_quantile[lower_limit])) + '_' + str(int(feature_quantile[upper_limit])))
    if duplicates==True:
      return [f'{int(feature_quantile[quantile_split[i]])}_{int(feature_quantile[quantile_split[i+1]])}' for i in range(len(quantile_split)-2)]
    else:
      return [f'{int(feature_quantile[quantile_split[i]])}_{int(feature_quantile[quantile_split[i+1]])}' for i in range(len(quantile_split)-1)]

In [187]:
def quantile_split(n_bins, feature, duplicates='raise'):
    split = np.linspace(0,1,n_bins+1)
    feature_split = feature.quantile(split)

    if duplicates == 'drop':
        labels = [f'{round(feature_split[split[i]])}_{round(feature_split[split[i+1]])}' for i in range(n_bins-1)]
    else:
        labels = [f'{round(feature_split[split[i]])}_{round(feature_split[split[i+1]])}' for i in range(n_bins)]

    feature_quantile = pd.qcut(feature, q=split, duplicates='drop', labels=labels)

    return feature_quantile

In [188]:
df_country = pd.read_csv('IpAddress_to_Country.csv')
df_country.head()

,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


In [189]:
df = pd.read_csv('Fraud_Data.csv')
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


#Merge 2 datasets by ip_address

In [190]:
# Create an IntervalIndex using lower_bound_ip_address and upper_bound_ip_address
idx = pd.IntervalIndex.from_arrays(df_country['lower_bound_ip_address'], df_country['upper_bound_ip_address'], closed='both')
country = df_country.iloc[idx.get_indexer(df['ip_address'])]['country']

df['country'] = country.values
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,country
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0,Japan
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0,United States
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1,United States
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0,Australia
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0,United States


In [191]:
df.isnull().sum()

user_id           0
signup_time       0
purchase_time     0
purchase_value    0
device_id         0
source            0
browser           0
sex               0
age               0
ip_address        0
class             0
country           0
dtype: int64

Country :
* count_country : total count of purchases
* category_country : qcut by using quantile (0, 0.25, 0.5, 0.75, 1)

In [192]:
df['count_country'] = df['country'].map(df['country'].value_counts())
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,country,count_country
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0,Japan,7306
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0,United States,58049
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1,United States,58049
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0,Australia,23810
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0,United States,58049


In [193]:
# Create a new column 'category_country' and categorize the count of occurrences of each country into intervals
df['category_country'] = quantile_split(n_bins=5, feature=df['count_country'], duplicates='drop')

In [194]:
df['category_country'].value_counts()

23810_58049    58049
2961_12038     37778
1_2961         31475
12038_23810    23810
Name: category_country, dtype: int64

In [195]:
# Count the occurrences of unique IP addresses where
# class is equal to 1 (fraudulent transactions)
df.loc[df['class']==1]['ip_address'].value_counts()

3.484934e+08    19
3.874758e+09    19
2.050964e+09    19
1.502818e+09    19
5.760609e+08    19
                ..
2.337532e+09     1
1.477500e+09     1
2.618542e+09     1
3.816468e+09     1
3.451155e+09     1
Name: ip_address, Length: 7277, dtype: int64

In [196]:
# Count the occurrences of each unique IP address and map it to the 'ip_address' column

df['count_ip'] = df['ip_address'].map(df['ip_address'].value_counts())
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,country,count_country,category_country,count_ip
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0,Japan,7306,2961_12038,1
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0,United States,58049,23810_58049,1
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1,United States,58049,23810_58049,12
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0,Australia,23810,12038_23810,1
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0,United States,58049,23810_58049,1


#Transform signup_time, purchase_time to datetime

Define online-time based on sigup_time and purchase_time

In [197]:
df[['signup_time', 'purchase_time']] = df[['signup_time', 'purchase_time']].apply(pd.to_datetime, errors='coerce')
df['online_time'] = (df['purchase_time'] - df['signup_time'])

# Calculate the time spent online for each user in minutes
df['online_time'] = df['online_time'].dt.seconds // 60


All purchase with online time in first minute is fraud. There's no reason that someone pays without seeing products.

## Classification model to predict fraud

In [198]:
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,country,count_country,category_country,count_ip,online_time
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0,Japan,7306,2961_12038,1,231
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0,United States,58049,23810_58049,1,299
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1,United States,58049,23810_58049,12,0
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0,Australia,23810,12038_23810,1,1001
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0,United States,58049,23810_58049,1,691


In [199]:
df.drop(['user_id', 'count_country', 'count_ip', 'device_id', 'category_country'], axis=1, inplace=True)

In [200]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   signup_time     151112 non-null  datetime64[ns]
 1   purchase_time   151112 non-null  datetime64[ns]
 2   purchase_value  151112 non-null  int64         
 3   source          151112 non-null  object        
 4   browser         151112 non-null  object        
 5   sex             151112 non-null  object        
 6   age             151112 non-null  int64         
 7   ip_address      151112 non-null  float64       
 8   class           151112 non-null  int64         
 9   country         151112 non-null  object        
 10  online_time     151112 non-null  int64         
dtypes: datetime64[ns](2), float64(1), int64(4), object(4)
memory usage: 12.7+ MB


In [201]:
df['country'].unique()

array(['Japan', 'United States', 'Australia', 'Canada', 'China', 'Brazil',
       'India', 'Argentina', 'United Kingdom', 'South Africa', 'Germany',
       'Mexico', 'Sweden', 'Korea Republic of', 'Peru', 'Portugal',
       'Bangladesh', 'France', 'Taiwan; Republic of China (ROC)',
       'Russian Federation', 'Belgium', 'Denmark', 'Netherlands',
       'Iran (ISLAMIC Republic Of)', 'Spain', 'Gabon', 'Saudi Arabia',
       'Hong Kong', 'Georgia', 'Syrian Arab Republic', 'Turkey',
       'New Zealand', 'European Union', 'Ukraine', 'Austria', 'Israel',
       'Malaysia', 'Greece', 'Italy', 'Norway', 'Poland', 'Venezuela',
       'Thailand', 'Chile', 'Morocco', 'Finland', 'Hungary', 'Indonesia',
       'Colombia', 'Ecuador', 'Lithuania', 'Switzerland', 'Viet Nam',
       'Nigeria', 'Egypt', 'Seychelles', 'Kazakhstan', 'Kenya',
       'Moldova Republic of', 'Trinidad and Tobago', 'Qatar', 'Bolivia',
       'Bulgaria', 'Romania', 'Croatia (LOCAL Name: Hrvatska)', 'Cyprus',
       'Czech Rep

In [202]:
source_mapping = {
    'SEO': 0,
    'Ads': 1,
    'Direct': 2
}
df['source'] = df['source'].map(source_mapping)

sex_mapping = {
    'M': 0,
    'F': 1,
}
df['sex'] = df['sex'].map(sex_mapping)

browser_mapping = {
    'Chrome': 0,
    'IE': 1,
    'Safari': 2,
    'FireFox': 3,
    'Opera': 4,
}
df['browser'] = df['browser'].map(browser_mapping)

In [203]:
country_mapping = {
    'Japan': 0, 'United States': 1, 'Australia': 2, 'Canada': 3, 'China': 4, 'Brazil': 5,
    'India': 6, 'Argentina': 7, 'United Kingdom': 8, 'South Africa': 9, 'Germany': 10,
    'Mexico': 11, 'Sweden': 12, 'Korea Republic of': 13, 'Peru': 14, 'Portugal': 15,
    'Bangladesh': 16, 'France': 17, 'Taiwan; Republic of China (ROC)': 18,
    'Russian Federation': 19, 'Belgium': 20, 'Denmark': 21, 'Netherlands': 22,
    'Iran (ISLAMIC Republic Of)': 23, 'Spain': 24, 'Gabon': 25, 'Saudi Arabia': 26,
    'Hong Kong': 27, 'Georgia': 28, 'Syrian Arab Republic': 29, 'Turkey': 30,
    'New Zealand': 31, 'European Union': 32, 'Ukraine': 33, 'Austria': 34, 'Israel': 35,
    'Malaysia': 36, 'Greece': 37, 'Italy': 38, 'Norway': 39, 'Poland': 40, 'Venezuela': 41,
    'Thailand': 42, 'Chile': 43, 'Morocco': 44, 'Finland': 45, 'Hungary': 46, 'Indonesia': 47,
    'Colombia': 48, 'Ecuador': 49, 'Lithuania': 50, 'Switzerland': 51, 'Viet Nam': 52,
    'Nigeria': 53, 'Egypt': 54, 'Seychelles': 55, 'Kazakhstan': 56, 'Kenya': 57,
    'Moldova Republic of': 58, 'Trinidad and Tobago': 59, 'Qatar': 60, 'Bolivia': 61,
    'Bulgaria': 62, 'Romania': 63, 'Croatia (LOCAL Name: Hrvatska)': 64, 'Cyprus': 65,
    'Czech Republic': 66, 'Algeria': 67, 'Kyrgyzstan': 68, 'Singapore': 69, 'Guam': 70,
    'United Arab Emirates': 71, 'Paraguay': 72, 'Tunisia': 73, 'Dominican Republic': 74,
    'Pakistan': 75, 'Malta': 76, 'Nicaragua': 77, 'Estonia': 78, 'Mozambique': 79,
    'Namibia': 80, 'Macedonia': 81, 'Costa Rica': 82, 'Cuba': 83, 'Ireland': 84,
    'Albania': 85, 'Oman': 86, 'Uruguay': 87, 'Lebanon': 88, 'Puerto Rico': 89,
    'Maldives': 90, 'Turkmenistan': 91, 'Barbados': 92, 'Iceland': 93, 'Philippines': 94,
    'Kuwait': 95, 'Panama': 96, 'New Caledonia': 97, 'Guatemala': 98, 'Ghana': 99,
    'Latvia': 100, 'Malawi': 101, 'Slovenia': 102, 'Senegal': 103,
    'Libyan Arab Jamahiriya': 104, 'Cambodia': 105, 'Belize': 106, 'Mauritius': 107,
    'Slovakia (SLOVAK Republic)': 108, 'Iraq': 109, 'El Salvador': 110,
    'Bosnia and Herzegowina': 111, 'Serbia': 112, 'Luxembourg': 113, 'Belarus': 114,
    "Cote D'ivoire": 115, 'Djibouti': 116, 'Armenia': 117, 'Sri Lanka': 118, 'Sudan': 119,
    'Rwanda': 120, 'Uzbekistan': 121, 'Jordan': 122, 'Bahrain': 123, 'Azerbaijan': 124,
    'South Sudan': 125, 'Virgin Islands (U.S.)': 126, 'Congo': 127, 'Angola': 128,
    'Uganda': 129, 'Jamaica': 130, 'Haiti': 131, 'Papua New Guinea': 132, 'Gibraltar': 133,
    'Cameroon': 134, 'Palestinian Territory Occupied': 135, 'Myanmar': 136, 'Nepal': 137,
    'Brunei Darussalam': 138, 'Zambia': 139, 'Saint Kitts and Nevis': 140, 'Reunion': 141,
    'Botswana': 142, 'Dominica': 143, 'Burkina Faso': 144, 'Montenegro': 145, 'Macau': 146,
    'Tanzania United Republic of': 147, 'Faroe Islands': 148, 'Zimbabwe': 149,
    'Honduras': 150, 'Monaco': 151, 'Congo The Democratic Republic of The': 152,
    'Cayman Islands': 153, 'Niger': 154, 'Antigua and Barbuda': 155, 'Lesotho': 156,
    'Fiji': 157, 'Mongolia': 158, 'Afghanistan': 159, 'Bhutan': 160, 'Bermuda': 161, 'Curacao': 162,
    'Ethiopia': 163, 'Vanuatu': 164, "Lao People's Democratic Republic": 165,
    'British Indian Ocean Territory': 166, 'Bahamas': 167, 'Madagascar': 168,
    'Bonaire; Sint Eustatius; Saba': 169, 'Liechtenstein': 170, 'Gambia': 171,
    'Benin': 172, 'Cape Verde': 173, 'Tajikistan': 174, 'Saint Martin': 175, 'Yemen': 176,
    'San Marino': 177, 'Burundi': 178, 'Nauru': 179, 'Guadeloupe': 180
}

# df['country'] = df['country'].map(browser_mapping)


unique_countries = df['country'].unique()

# Create the mapping dictionary
country_mapping = {country: idx for idx, country in enumerate(unique_countries)}
# Encode the countries
df['country'] = df['country'].map(country_mapping)

In [204]:
# Define columns to scale
columns_to_scale = ['purchase_value', 'age', 'online_time']

# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Fit and transform the selected columns
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

# Save the fitted scaler to a file
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [205]:
df.head()

,signup_time,purchase_time,purchase_value,source,browser,sex,age,ip_address,class,country,online_time
0,2015-02-24 22:55:49,2015-04-18 02:47:11,0.172414,0,0,0,0.362069,7.327584e+08,0,0,0.160528
1,2015-06-07 20:39:50,2015-06-08 01:38:54,0.048276,1,0,1,0.603448,3.503114e+08,0,1,0.207783
2,2015-01-01 18:52:44,2015-01-01 18:52:45,0.041379,0,4,0,0.603448,2.621474e+09,1,1,0.000000
3,2015-04-28 21:13:25,2015-05-04 13:54:50,0.241379,0,2,0,0.396552,3.840542e+09,0,2,0.695622
4,2015-07-21 07:09:52,2015-09-09 18:40:53,0.206897,1,2,0,0.465517,4.155831e+08,0,1,0.480195


In [206]:
numeric_feats = ['purchase_value', 'age', 'online_time', ]
categorical_feats = ['source', 'sex', 'browser', 'country']
target = ['class']
df_model = df[numeric_feats + categorical_feats + target]

X = df[numeric_feats + categorical_feats]
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

In [207]:
X.head()

,purchase_value,age,online_time,source,sex,browser,country
0,0.172414,0.362069,0.160528,0,0,0,0
1,0.048276,0.603448,0.207783,1,1,0,1
2,0.041379,0.603448,0.000000,0,0,4,1
3,0.241379,0.396552,0.695622,0,0,2,2
4,0.206897,0.465517,0.480195,1,0,2,1


In [208]:
#feedforward neural network (FNN)
# Define the model
model = Sequential()
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

# Evaluate the model
_, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
_, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Train Accuracy: {train_accuracy*100:.2f}%")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

Epoch 1/10
4250/4250 [==============================] - 11s 2ms/step - loss: 0.2615 - accuracy: 0.9162
Epoch 2/10
4250/4250 [==============================] - 9s 2ms/step - loss: 0.2122 - accuracy: 0.9422
Epoch 3/10
4250/4250 [==============================] - 9s 2ms/step - loss: 0.2007 - accuracy: 0.9466
Epoch 4/10
4250/4250 [==============================] - 9s 2ms/step - loss: 0.1956 - accuracy: 0.9492
Epoch 5/10
4250/4250 [==============================] - 9s 2ms/step - loss: 0.1925 - accuracy: 0.9510
Epoch 6/10
4250/4250 [==============================] - 9s 2ms/step - loss: 0.1908 - accuracy: 0.9514
Epoch 7/10
4250/4250 [==============================] - 9s 2ms/step - loss: 0.1888 - accuracy: 0.9523
Epoch 8/10
4250/4250 [==============================] - 9s 2ms/step - loss: 0.1886 - accuracy: 0.9525
Epoch 9/10
4250/4250 [==============================] - 10s 2ms/step - loss: 0.1880 - accuracy: 0.9526
Epoch 10/10
4250/4250 [==============================] - 9s 2ms/step - loss: 0.1

In [209]:
# Predictions
y_pred = model.predict(X_test)
y_pred_classes = (y_pred > 0.5).astype("int32")

# Classification Report
print(classification_report(y_test, y_pred_classes))


473/473 [==============================] - 1s 2ms/step
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     13697
           1       0.95      0.53      0.68      1415

    accuracy                           0.95     15112
   macro avg       0.95      0.76      0.83     15112
weighted avg       0.95      0.95      0.95     15112



In [210]:
model.save("fd.h5")

C:\Users\lbasa\anaconda3\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [211]:
# load model
savedModel=load_model('fd.h5')
savedModel.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_11 (Dense)            (None, 64)                512       
                                                                 
 dense_12 (Dense)            (None, 32)                2080      
                                                                 
 dense_13 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2625 (10.25 KB)
Trainable params: 2625 (10.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [212]:
# # Reshape the input data to be 3-dimensional for LSTM
# X_train = np.reshape(X_train, (X_train.shape[0], 1, X_train.shape[1]))
# X_test = np.reshape(X_test, (X_test.shape[0], 1, X_test.shape[1]))

# # Define the model
# model = Sequential()
# model.add(LSTM(100, input_shape=(X_train.shape[1], X_train.shape[2])))
# model.add(Dense(1, activation='sigmoid'))

# # Compile the model
# model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# # Train the model
# model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

# # Evaluate the model
# _, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
# _, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

# print(f"Train Accuracy: {train_accuracy*100:.2f}%")
# print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# # Predictions
# y_pred = model.predict(X_test)
# y_pred_classes = (y_pred > 0.5).astype("int32")

# # Classification Report
# print(classification_report(y_test, y_pred_classes))